In [1]:
from pandas_gbq import read_gbq
from pandas_gbq.auth import get_credentials
#import pandas_gbq
import pandas as pd
from pathlib import Path
from datetime import datetime
import time
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import os
import warnings
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [2]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
FOLDER_PATH= "data_entries/TRAMA BS/1. Desgravamen_TC_BN/2025/" 
LOCAL_PATH= "C:/TRAMAS BANCA SEGUROS/BANCO DE LA NACION/ALTAS 2024" 
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_BS_DESGRAVAMEN_TC_BN_2024"
TABLE_CONTROL_ID= "CONTROL_tramas_BN"

In [3]:
# Autenticarse en BIGQUERY
credentials, _ = get_credentials()

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

In [4]:
#espacios_col = [(47, 51), (27, 47), (114, 115), (111,114), (3983, 3991), (3991, 3999), (3655, 3670), (3685, 3700)] # Para pagos
espacios_col = [(47, 51), (27, 47), (114, 115), (111,114), (3639, 3647), (3647, 3655), (3655, 3670), (3685, 3700)] # Para altas

column_names = ["Codigo Producto", "Certificado canal", "Tipo movimiento", "Moneda",
                "Fecha inicio", "Fecha fin", "Suma Asegurada", "Prima Bruta"
                ]

# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")
    return row


def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan
        return float(valor_str) / 100
    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo


# Convertir fechas string a tipo datetime
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha


def buscar_archivo(filename_txt: str) -> bool:
    # Verificar si existe la tabla de control
    table_control_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONTROL_ID)
    if not tabla_existe(table_control_ref):
        return False
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            SELECT COUNT(*) AS count
            FROM `{table_control_ref}`
            WHERE ARCHIVO_TXT = @archivo_txt
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", filename_txt)]
        )

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df["count"].iloc[0] > 0


#### FUNCION PARA COMPROBAR SI LA TABLA ESTA CREADA
def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise

#### FUNCION PARA GUARDAR UN DATASET EN UNA TABLA DE BIGQUERY
def Guardar_en_BigQuery(data, dataset_id, table_id):
    table_ref = bigquery_client.dataset(dataset_id).table(table_id)
    job_config = bigquery.LoadJobConfig()
    if tabla_existe(table_ref):
        # Abre la tabla para agregar registros
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_APPEND
    else:
        tabla_tramas = bigquery.Table(table_ref)
        tabla_tramas = bigquery_client.create_table(tabla_tramas)
        job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
        job_config.autodetect = False
        print(f'   ℹ️ ... Se ha creado la tabla: {table_id} en el dataset: {dataset_id} ...')
    job = bigquery_client.load_table_from_dataframe(data, table_ref, job_config=job_config)
    job.result()
    return

#### PROCESAR ARCHIVOS
carpeta= Path(LOCAL_PATH)
for archivo in carpeta.iterdir():
    if archivo.is_file() and archivo.suffix.lower() == ".txt":
        file_name= archivo.name
        if not buscar_archivo(file_name):
            print(f"⏳ ... PROCESANDO ARCHIVO: {file_name} ... ⏳")
            with open(archivo, "r", encoding="latin-1") as file:
                lista_lineas = file.readlines()[1:]
                df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
                df_tramas.columns = (df_tramas.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                                        )
                df_tramas["PRIMA_BRUTA"] = df_tramas["PRIMA_BRUTA"].apply(convertir_prima)
                df_tramas["SUMA_ASEGURADA"] = df_tramas["SUMA_ASEGURADA"].apply(convertir_prima)
                df_tramas["FECHA_INICIO"] = Convertir_fecha(df_tramas["FECHA_INICIO"]).dt.date
                df_tramas["FECHA_FIN"] = Convertir_fecha(df_tramas["FECHA_FIN"]).dt.date
                df_tramas["NOMBRE_ARCHIVO"] = file_name

                fecha_carga = datetime.now()
                df_control = pd.DataFrame([{"ARCHIVO_TXT": file_name, "FECHA_CARGA": fecha_carga}])
                Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID)
                Guardar_en_BigQuery(df_control, DATASET_ID, TABLE_CONTROL_ID)
                print(f"✅ ARCHIVO {file_name} CARGADO ...")
        else:
            print(f"⚠️ EL ARCHIVO {file_name} YA FUE CARGADO ANTERIORMENTE ...")



⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20231230-001.TXT ... ⏳
   ℹ️ ... Se ha creado la tabla: TRAMAS_BS_DESGRAVAMEN_TC_BN_2024 en el dataset: produccion ...
✅ ARCHIVO 0100030595_0144001-20231230-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20240102-001.TXT ... ⏳
✅ ARCHIVO 0100030595_0144001-20240102-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20240103-001.TXT ... ⏳
✅ ARCHIVO 0100030595_0144001-20240103-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20240104-001.TXT ... ⏳
✅ ARCHIVO 0100030595_0144001-20240104-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20240105-001.TXT ... ⏳
✅ ARCHIVO 0100030595_0144001-20240105-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20240106-001.TXT ... ⏳
✅ ARCHIVO 0100030595_0144001-20240106-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHIVO: 0100030595_0144001-20240108-001.TXT ... ⏳
✅ ARCHIVO 0100030595_0144001-20240108-001.TXT CARGADO ...
⏳ ... PROCESANDO ARCHI

## TEST

In [8]:
trama= "C:/Datos/TRAMA_ALTA_BN_muestra.txt"

In [9]:
with open(trama, "r", encoding="latin-1") as file:
    lista_lineas = file.readlines()[1:]

In [10]:
len(lista_lineas)

1000

In [32]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])
df_tramas.columns = (df_tramas.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
                        )

In [33]:
df_tramas.head(3)

,CODIGO_PRODUCTO,CERTIFICADO_CANAL,TIPO_MOVIMIENTO,MONEDA,FECHA_INICIO,FECHA_FIN,SUMA_ASEGURADA,PRIMA_BRUTA,TRAMA_ORIGINAL
0,0144,00000000000000000000,0,SOL,20200306,20370915,000000000000000,000000000000560,0000000001000000000101440010000000000000000000...
1,0144,00000000000000000000,0,SOL,20191226,20470527,000000000000000,000000000000560,0000000002000000000201440010000000000000000000...
2,0144,00000000000000000000,0,SOL,20210903,20301014,000000000000000,000000000000560,0000000003000000000301440010000000000000000000...


In [34]:
df_tramas["PRIMA_BRUTA"] = df_tramas["PRIMA_BRUTA"].apply(convertir_prima)
df_tramas["SUMA_ASEGURADA"] = df_tramas["SUMA_ASEGURADA"].apply(convertir_prima)
df_tramas["FECHA_INICIO"] = Convertir_fecha(df_tramas["FECHA_INICIO"]).dt.date
df_tramas["FECHA_FIN"] = Convertir_fecha(df_tramas["FECHA_FIN"]).dt.date

In [35]:
df_tramas.head(3)

,CODIGO_PRODUCTO,CERTIFICADO_CANAL,TIPO_MOVIMIENTO,MONEDA,FECHA_INICIO,FECHA_FIN,SUMA_ASEGURADA,PRIMA_BRUTA,TRAMA_ORIGINAL
0,0144,00000000000000000000,0,SOL,2020-03-06,2037-09-15,0.0,5.6,0000000001000000000101440010000000000000000000...
1,0144,00000000000000000000,0,SOL,2019-12-26,2047-05-27,0.0,5.6,0000000002000000000201440010000000000000000000...
2,0144,00000000000000000000,0,SOL,2021-09-03,2030-10-14,0.0,5.6,0000000003000000000301440010000000000000000000...


In [36]:
Guardar_en_BigQuery(df_tramas, DATASET_ID, TABLE_ID)

   ℹ️ ... Se ha creado la tabla: TRAMAS_BS_DESGRAVAMEN_TC_BN-2 en el dataset: produccion ...


In [24]:
df_tramas.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CODIGO_PRODUCTO    1000 non-null   str    
 1   CERTIFICADO_CANAL  1000 non-null   str    
 2   TIPO_MOVIMIENTO    1000 non-null   str    
 3   MONEDA             1000 non-null   str    
 4   FECHA_INICIO       1000 non-null   object 
 5   FECHA_FIN          1000 non-null   object 
 6   SUMA_ASEGURADA     1000 non-null   float64
 7   PRIMA_BRUTA        1000 non-null   float64
 8   TRAMA_ORIGINAL     1000 non-null   str    
dtypes: float64(2), object(2), str(5)
memory usage: 70.4+ KB
